## 1077. Project Employees III
### Table: Project

| Column Name | Type |
|-------------|------|
| project_id  | int  |
| employee_id | int  |

(project_id, employee_id) is the primary key of this table.  
employee_id is a foreign key to Employee table.

### Table: Employee

| Column Name      | Type    |
|------------------|---------|
| employee_id      | int     |
| name             | varchar |
| experience_years | int     |

employee_id is the primary key of this table.

Write an SQL query that reports the most experienced employees in each project. In case of a tie, report all employees with the maximum number of experience years.

The query result format is in the following example:

### Project table:

| project_id | employee_id |
|------------|-------------|
| 1          | 1           |
| 1          | 2           |
| 1          | 3           |
| 2          | 1           |
| 2          | 4           |

### Employee table:

| employee_id | name   | experience_years |
|-------------|--------|------------------|
| 1           | Khaled | 3                |
| 2           | Ali    | 2                |
| 3           | John   | 3                |
| 4           | Doe    | 2                |

### Result table:

| project_id | employee_id |
|------------|-------------|
| 1          | 1           |
| 1          | 3           |
| 2          | 1           |

Both employees with id 1 and 3 have the most experience among the employees of the first project.  
For the second project, the employee with id 1 has the most experience.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

# Define schemas
project_schema = StructType([
    StructField("project_id", IntegerType(), True),
    StructField("employee_id", IntegerType(), True)
])

employee_schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("experience_years", IntegerType(), True)
])

# Sample data
project_data = [
    (1, 1), (1, 2), (1, 3),
    (2, 1), (2, 4)
]

employee_data = [
    (1, "Khaled", 3),
    (2, "Ali", 2),
    (3, "John", 3),
    (4, "Doe", 2)
]

# Create DataFrames
project_df = spark.createDataFrame(project_data, schema=project_schema)
employee_df = spark.createDataFrame(employee_data, schema=employee_schema)

# Register temp views
project_df.createOrReplaceTempView("Project")
employee_df.createOrReplaceTempView("Employee")


In [0]:
%sql
with cte as (
  select project_id , e.employee_id , rank()over(partition by project_id order by experience_years desc ) as m_e  from Project p left join Employee e on p.employee_id = e.employee_id
)
select project_id , employee_id  from cte where m_e = 1 order by project_id , employee_id asc

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

pdf = project_df.selectExpr("project_id as project_id", "employee_id as p_employee_id")
edf = employee_df.selectExpr(
    "employee_id as e_employee_id",
    "name as name",
    "experience_years as experience_years",
)
pe_df = pdf.join(edf, col("p_employee_id") == col("e_employee_id"), "left")
win_spec = Window.partitionBy(col("project_id")).orderBy(col("experience_years").desc())
rank = rank().over(win_spec)
pe_df.withColumn("rnk", rank).filter(col("rnk") == 1).selectExpr(
    "project_id", "p_employee_id as employee_id"
).display()

In [0]:

# SQL logic
spark.sql("""
WITH joined AS (
    SELECT 
        p.project_id,
        e.employee_id,
        e.experience_years
    FROM Project p
    JOIN Employee e ON p.employee_id = e.employee_id
),
ranked AS (
    SELECT 
        project_id,
        employee_id,
        experience_years,
        RANK() OVER (PARTITION BY project_id ORDER BY experience_years DESC) AS rnk
    FROM joined
)
SELECT project_id, employee_id
FROM ranked
WHERE rnk = 1
""").createOrReplaceTempView("Result")

# Display result
display(spark.sql("SELECT * FROM Result"))